<a href="https://colab.research.google.com/github/TBGhorbanpour/Social-Awareness/blob/main/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from transformers import pipeline
import torch

# ==========================================
# 1. Configuration & Device Check
# ==========================================

#INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data_Paper/MainDB_Final_BERT_Ready.csv'
INPUT_CSV ='/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_Filtered_lemma_clean.csv'
OUTPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output.csv'

# Use GPU if available (Colab default), otherwise fallback to CPU
device = 0 if torch.cuda.is_available() else -1
print(f"🚀 Using device: {'GPU (CUDA)' if device == 0 else 'CPU'}")

# ==========================================
# 2. Load Data
# ==========================================
print("Loading dataset...")
df = pd.read_csv(INPUT_CSV)

# Ensure we have clean strings and handle NaNs
texts = df['FormalTweet'].fillna('').astype(str).tolist()

# ==========================================
# 3. Initialize Batched NER Pipeline
# ==========================================
print("Loading ParsBERT NER model...")
ner_pipeline = pipeline(
    "ner",
    model="HooshvareLab/bert-fa-base-uncased-ner-peyma",
    tokenizer="HooshvareLab/bert-fa-base-uncased-ner-peyma",
    device=device,
    aggregation_strategy="simple", # Automatically merges subwords into full words
    batch_size=16                  # Process 16 tweets at a time
)
# ==========================================
# 4. Run Inference (Fast!)
# ==========================================
print(f"Starting batched NER on {len(texts)} tweets... (This will take ~15-20 mins)")
ner_results = ner_pipeline(texts)

# ==========================================
# 5. Format and Save Results
# ==========================================
# Add the raw dictionary output to the dataframe
df['ner_entities_raw'] = ner_results

# Create a clean, readable list of (Word, Label) tuples for analysis
df['ner_entities'] = df['ner_entities_raw'].apply(
    lambda x: [(ent['word'], ent['entity_group']) for ent in x] if x else []
)

# Optional: Create a count column for quick filtering
df['ner_entity_count'] = df['ner_entities'].apply(len)

# Save to Drive
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Done! Saved batched NER results to {OUTPUT_CSV}")

# Display Preview
display(df[['FormalTweet', 'ner_entities', 'ner_entity_count']].head(5))

🚀 Using device: GPU (CUDA)
Loading dataset...
Loading ParsBERT NER model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased-ner-peyma
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting batched NER on 75455 tweets... (This will take ~15-20 mins)
✅ Done! Saved batched NER results to /content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output.csv


,FormalTweet,ner_entities,ner_entity_count
0,نان توموشی بارش بندر عالی میفهمند خاک هرمز ما...,"[(هرمز, B_LOC)]",1
1,سرنج معنی خاک سرخ وایب هرمز,"[(هرمز, B_LOC)]",1
2,خاک خاک جزیره ی هرمز,"[(جزیره, B_LOC), (ی هرمز, I_LOC)]",2
3,مست زمین خورده کاس هرمز خون دل نوشروان رفتند ...,"[(هرمز, B_PER)]",1
4,قطار بروی بندر عباس لنج بروی جزیره کیش قشم هر...,"[(بندر, B_LOC), (عباس, I_LOC), (جزیره, B_LOC),...",6


In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ==========================================
# 1. Configuration & Device Check
# ==========================================
INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output.csv'
OUTPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output2.csv'

# Force GPU usage if available (Colab T4/A100)
device = 0 if torch.cuda.is_available() else -1
print(f" Using device: {'GPU (CUDA)' if device == 0 else 'CPU (Warning: This will be slow!)'}")

# ==========================================
# 2. Load Data
# ==========================================
print("Loading dataset...")
df = pd.read_csv(INPUT_CSV)

# Ensure we have clean strings and handle NaNs safely
texts = df['FormalTweet'].fillna('').astype(str).tolist()
print(f"Loaded {len(texts)} tweets for NER processing.")

# ==========================================
# 3. Initialize Batched NER Pipeline
# ==========================================
print("Loading ParsBERT NER model...")
ner_pipeline = pipeline(
    "ner",
    model="HooshvareLab/bert-fa-base-uncased-ner-peyma",
    tokenizer="HooshvareLab/bert-fa-base-uncased-ner-peyma",
    device=device,
    aggregation_strategy="simple", # Automatically merges subwords into full words
    batch_size=16                  # Process 16 tweets at a time
)


# ==========================================
# 4. Run Inference (Fast!)
# ==========================================
print("Starting batched NER... (This will take ~10-20 mins on GPU)")
ner_results = ner_pipeline(texts)

# ==========================================
# 5. Format and Save Results
# ==========================================
print("Formatting results...")
# Add the raw dictionary output to the dataframe
df['ner_entities_raw'] = ner_results

# Create a clean, readable list of (Word, Label) tuples for analysis
df['decoded_entities'] = df['ner_entities_raw'].apply(
    lambda x: [(ent['word'], ent['entity_group']) for ent in x] if isinstance(x, list) else []
)

# Optional: Create a count column for quick filtering
df['ner_entity_count'] = df['decoded_entities'].apply(len)

# Save to Drive
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f" Done! Saved batched NER results to {OUTPUT_CSV}")

# Display Preview
display_cols = ['FormalTweet', 'decoded_entities', 'ner_entity_count']
display(df[display_cols].head(5))

 Using device: GPU (CUDA)
Loading dataset...
Loaded 75455 tweets for NER processing.
Loading ParsBERT NER model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased-ner-peyma
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting batched NER... (This will take ~10-20 mins on GPU)
Formatting results...
 Done! Saved batched NER results to /content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output2.csv


,FormalTweet,decoded_entities,ner_entity_count
0,نان توموشی بارش بندر عالی میفهمند خاک هرمز ما...,"[(هرمز, B_LOC)]",1
1,سرنج معنی خاک سرخ وایب هرمز,"[(هرمز, B_LOC)]",1
2,خاک خاک جزیره ی هرمز,"[(جزیره, B_LOC), (ی هرمز, I_LOC)]",2
3,مست زمین خورده کاس هرمز خون دل نوشروان رفتند ...,"[(هرمز, B_PER)]",1
4,قطار بروی بندر عباس لنج بروی جزیره کیش قشم هر...,"[(بندر, B_LOC), (عباس, I_LOC), (جزیره, B_LOC),...",6


In [ ]:
import pandas as pd

INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output2.csv'
OUTPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output3.csv'

data = pd.read_csv(INPUT_CSV)

# 1. Optimized extraction function
def extract_cities(named_entities):
    if not isinstance(named_entities, list):
        return []

    cities = []
    current_city = []

    for token, label in named_entities:
        # ParsBERT PEYMA uses 'B-LOC' and 'I-LOC'
        if 'LOC' in label.upper():
            # Clean up any stray BERT subword markers just in case
            clean_token = token.replace("##", "").strip()
            current_city.append(clean_token)
        else:
            # If the chain of location words breaks, save the city
            if current_city:
                cities.append("".join(current_city))
                current_city = []

    # Append the last city if the sentence ended with a location
    if current_city:
        cities.append("".join(current_city))

    return cities

# 2. Apply using List Comprehension
# FIX: Changed 'named_entities' to 'decoded_entities' to match the batched NER output
print("Extracting cities...")
data['cities'] = [extract_cities(ents) for ents in data['decoded_entities']]

# 3. Create a count column
data['city_count'] = data['cities'].apply(len)

# Check the first few rows
print(data[['decoded_entities', 'cities', 'city_count']].head())

# Save to CSV
data.to_csv(OUTPUT_CSV, sep=',', index=False, encoding='utf-8-sig')
print(f"✅ Saved to {OUTPUT_CSV}")

Extracting cities...
                                    decoded_entities cities  city_count
0                                [('هرمز', 'B_LOC')]     []           0
1                                [('هرمز', 'B_LOC')]     []           0
2          [('جزیره', 'B_LOC'), ('ی هرمز', 'I_LOC')]     []           0
3                                [('هرمز', 'B_PER')]     []           0
4  [('بندر', 'B_LOC'), ('عباس', 'I_LOC'), ('جزیره...     []           0
✅ Saved to /content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output3.csv


In [ ]:
import pandas as pd
import ast

INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output2.csv'
OUTPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output3.csv'

data = pd.read_csv(INPUT_CSV)

# 1. Optimized extraction function with spacing fix
def extract_cities(named_entities):
    # Convert string back to list if read from CSV
    if isinstance(named_entities, str):
        try:
            named_entities = ast.literal_eval(named_entities)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(named_entities, list):
        return []

    cities = []
    current_city = []

    for token, label in named_entities:
        if 'LOC' in str(label).upper() or 'LOCATION' in str(label).upper():
            # Clean up subword markers and strip whitespace
            clean_token = token.replace("##", "").strip()
            if clean_token:
                current_city.append(clean_token)
        else:
            if current_city:
                # FIX: Use " ".join() to add spaces between Persian words
                cities.append(" ".join(current_city))
                current_city = []

    if current_city:
        cities.append(" ".join(current_city))

    return cities

# 2. Apply extraction
print("Extracting cities...")
data['cities'] = [extract_cities(ents) for ents in data['decoded_entities']]

# 3. Create a count column
data['city_count'] = data['cities'].apply(len)

# Quick diagnostic check
print(f"Total rows with cities found: {data['city_count'].sum()}")
print(data[['decoded_entities', 'cities', 'city_count']].head())

# Save to CSV
data.to_csv(OUTPUT_CSV, sep=',', index=False, encoding='utf-8-sig')
print(f"✅ Saved to {OUTPUT_CSV}")

Extracting cities...
Total rows with cities found: 36743
                                    decoded_entities  \
0                                [('هرمز', 'B_LOC')]   
1                                [('هرمز', 'B_LOC')]   
2          [('جزیره', 'B_LOC'), ('ی هرمز', 'I_LOC')]   
3                                [('هرمز', 'B_PER')]   
4  [('بندر', 'B_LOC'), ('عباس', 'I_LOC'), ('جزیره...   

                           cities  city_count  
0                          [هرمز]           1  
1                          [هرمز]           1  
2                  [جزیره ی هرمز]           1  
3                              []           0  
4  [بندر عباس جزیره کیش قشم هرمز]           1  
✅ Saved to /content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_NER_Output3.csv


In [ ]:
data.head(5)

,Column1,ID,Date,Tweet,URL,Likes,Retweet,Replies,Source,FormalTweet,...,fully_cleaned_text,tokenized_tweet,lemmatized_tweet,list_of_lists,ner_entities_raw,ner_entities,ner_entity_count,decoded_entities,cities,city_count
0,1,1.560507e+18,2022-08-19 10:30:12,@bbrdiya تاحالا نون توموشی به کسی که اولین بار...,https://twitter.com/yourpenpalfar/status/15605...,1,0,0,iPhone,نان توموشی بارش بندر عالی میفهمند خاک هرمز ما...,...,نان توموشی بارش بندر عالی میفهمند خاک هرمز ماه...,نان توموشی بارش بندر عالی میفهمند خاک هرمز ماه...,ﻧﺎﻥ ﺗﻮﻣﻮﺷﯽ ﺑﺎﺭﺵ ﺑﻨﺪﺭ ﻋﺎﻟﯽ ﻣﯿﻔﻬﻤﻨﺪ ﺧﺎﮎ ﻫﺮﻣﺰ ﻣﺎﻫ...,"['نان', 'توموشی', 'بارش', 'بندر', 'عالی', 'میف...","[{'entity_group': 'B_LOC', 'score': np.float32...","[('هرمز', 'B_LOC')]",1,"[('هرمز', 'B_LOC')]",[هرمز],1
1,2,1.560269e+18,2022-08-18 18:46:43,@zahrahp1998 سرنج\nبه معنی خاک سرخ\nوایب هرمز ...,https://twitter.com/alipaydarr/status/15602694...,0,0,1,iPad,سرنج معنی خاک سرخ وایب هرمز,...,سرنج معنی خاک سرخ وایب هرمز,سرنج معنی خاک سرخ وایب هرمز,ﺳﺮﻧﺞ ﻣﻌﻨﺎ ﺧﺎﮎ ﺳﺮﺥ ﻭﺍﯾﺐ ﻫﺮﻣﺰ,"['سرنج', 'معنا', 'خاک', 'سرخ', 'وایب', 'هرمز']","[{'entity_group': 'B_LOC', 'score': np.float32...","[('هرمز', 'B_LOC')]",1,"[('هرمز', 'B_LOC')]",[هرمز],1
2,5,1.558507e+18,2022-08-13 22:04:51,@sina_mim من خاک آوردم، خاک جزیره ی هرمز,https://twitter.com/Abi_in_dream/status/155850...,1,0,0,iPhone,خاک خاک جزیره ی هرمز,...,خاک خاک جزیره هرمز,خاک خاک جزیره هرمز,ﺧﺎﮎ ﺧﺎﮎ ﺟﺰﯾﺮﻩ ﻫﺮﻣﺰ,"['خاک', 'خاک', 'جزیره', 'هرمز']","[{'entity_group': 'B_LOC', 'score': np.float32...","[('جزیره', 'B_LOC'), ('ی هرمز', 'I_LOC')]",2,"[('جزیره', 'B_LOC'), ('ی هرمز', 'I_LOC')]",[جزیره ی هرمز],1
3,6,1.558160e+18,2022-08-12 23:05:28,مست است زمین زیرا خورده‌ست به‌ جایِ می\nدر کاس...,https://twitter.com/Papyrustw/status/155816025...,18,3,0,App,مست زمین خورده کاس هرمز خون دل نوشروان رفتند ...,...,مست زمین خورده کاس هرمز خون دل نوشروان رفتند ت...,مست زمین خورده کاس هرمز خون دل نوشروان رفتند ت...,ﻣﺴﺖ ﺯﻣﯿﻦ ﮐﺎﺱ ﻫﺮﻣﺰ ﺧﻮﻥ ﺩﻝ ﻧﻮﺷﺮﻭ ﺗﺎﺝ ﻭﺭﺍﻥ ﺷﮑﻢ ﺧﺎ...,"['مست', 'زمین', 'کاس', 'هرمز', 'خون', 'دل', 'ن...","[{'entity_group': 'B_PER', 'score': np.float32...","[('هرمز', 'B_PER')]",1,"[('هرمز', 'B_PER')]",[],0
4,7,1.557824e+18,2022-08-12 00:51:18,@ArtemisPines میتونی یکاری کنی \nبا قطار بری ب...,https://twitter.com/ali_sagdast/status/1557824...,0,0,1,Android,قطار بروی بندر عباس لنج بروی جزیره کیش قشم هر...,...,قطار بروی بندر عباس لنج بروی جزیره کیش قشم هرم...,قطار بروی بندر عباس لنج بروی جزیره کیش قشم هرم...,ﻗﻄﺎﺭ ﺑﻨﺪﺭ ﻋﺒﺎﺱ ﻟﻨﺞ ﺟﺰﯾﺮﻩ ﮐﯿﺶ ﻗﺸﻢ ﻫﺮﻣﺰ ﺧﺎﮎ ﺭﻧﮕﯽ...,"['قطار', 'بندر', 'عباس', 'لنج', 'جزیره', 'کیش'...","[{'entity_group': 'B_LOC', 'score': np.float32...","[('بندر', 'B_LOC'), ('عباس', 'I_LOC'), ('جزیره...",6,"[('بندر', 'B_LOC'), ('عباس', 'I_LOC'), ('جزیره...",[بندر عباس جزیره کیش قشم هرمز],1
